# Tutorial 5: When does a random train–test split mislead us?

## Briefing for the responsible study group

| Planning item | Allocation |
|---|---|
| Presentation date | Tuesday 24 November 2026 |
| Prerequisite lecture | Forecasting & Prediction, 10 November 2026; Introduction as context |
| Preparation time after prerequisite | 9 full weekdays, excluding lecture and tutorial dates |
| Format | 15-minute presentation + approximately 15-minute discussion |

**Lecture connection:** Forecasting slides 29–40: holdout testing and cross-validation; Introduction slides 29–34: data over time.

This is your group's working notebook. It replaces the requirement to augment a section of an existing course notebook. The supplied cells establish a starting point; your group develops and teaches the investigation.

### Your teaching challenge

The lecture explains why evaluation needs observations not used for training. Your task is to show why that requirement alone is insufficient: the validation design must also represent how the model will be used. A random split can ask a different question from forecasting a later period.

Use the supplied synthetic series of independent cases observed over time. A change in market conditions shifts the target after a specified point. You receive a development period and a separate, later future period. The timestamp is provided for splitting and diagnostics; keep it out of the model inputs in the core experiment so the comparison isolates validation design under an omitted time effect.

### Core investigation

1. Plot the target over time and explain the supplied data-generating process. State the intended deployment task: fit on available past cases and predict later cases.
2. Compare random and chronological validation splits within the development period. Use the same model type, feature set and training/validation sample sizes. The model-building helper is supplied; your group designs the evaluation and interpretation.
3. Repeat random splitting with a few seeds. Compare its estimates with chronological validation. Explain how the compositions and time directions of the splits differ; this is not a controlled change of split labels on otherwise identical training data.
4. After fixing your procedure, refit one final model on **all development observations** and evaluate it once on the untouched future period. Compare the earlier validation estimates with this prospective result, while acknowledging that the final model has more training data and a different training composition.
5. Switch off the shift as a contrasting condition. Explain why random splitting can be appropriate for exchangeable observations and why chronological splitting is not automatically superior for every task.

### What classmates should gain

They should distinguish interpolation within a mixed historical population from prediction in a later environment. Explain the distinction between temporal distribution shift and explicit target leakage. The supplied data have shift; no feature directly contains the answer. Random validation nevertheless allows observations later than some validation cases into training.

### Boundaries and support

The generator, split indices and untrained model helper are supplied to keep the workload comparable. Do not add grouped validation, nested CV, drift detection or extensive tuning. The future set is a final check, not a resource for choosing settings. Use the disclosed generator to plan the contrasting condition; do not repeatedly inspect the future errors to engineer a preferred conclusion.

## How to turn your investigation into a tutorial

Your group has **15 minutes to present**, followed by approximately **15 minutes of discussion** with the instructor and class. Your audience has attended the prerequisite lecture but has not done your investigation. Help them understand one important question through a focused, reproducible demonstration.

**Required contribution.** Complete the core investigation below, explain why the results occur, and discuss at least one limitation or contrasting condition. Select two or three main figures or tables that make your argument clear. A useful tutorial connects the question, experimental design, evidence and conclusion; a sequence of unexplained code cells is insufficient. You are not expected to build a production system, implement algorithms from scratch, collect external data or run an extensive model search.

**What to submit.** Submit your completed `.ipynb` notebook with explanations and reproducible code, plus any slides or other presentation material you use. Follow the instructor's announced submission deadline. Keep the supplied briefing, but edit, reorganize or replace the working sections as needed. Identify substantial changes to supplied helpers. Record package versions, seeds and relevant settings; restart the kernel and run all cells before submission. Clearly distinguish supplied material from your group's contribution.

**Presentation format is your choice.** You may present from the notebook, create slides, render to HTML, develop an interactive demonstration or combine these. Make the selected format work within 15 minutes and have a fallback for live demonstrations. Creativity earns credit when it improves understanding, engagement or the quality of the investigation. Technical spectacle should be avoided. A clear static plot can be excellent.

**AI assistance is allowed.** You may use AI to help develop code, explanations and visualizations. Your group remains responsible for correctness and must be able to explain the submitted work. Include a short account of the tools used, their main contributions, and how you checked those contributions. Verify at least one important result independently—for example, with a hand calculation, an analytic special case or a comparison with a trusted library function. A long prompt transcript is not required. Using AI is optional and is not itself rewarded.

**Assessment.** This component contributes **15% of the course grade**. Within the tutorial, the criteria are **conceptual understanding (40%)**, **added value for classmates (40%)**, and **quality of communication (20%)**. Added value comes from the investigation and how it develops understanding beyond the lecture and the supplied starter cells. Optional extensions are not required for an excellent grade. All groups should prioritize a sound core investigation over breadth.

**Suggested presentation rhythm:** about 3 minutes for the question and essential concepts, 8 minutes for the demonstration and evidence, and 4 minutes for interpretation and limitations. Prepare one question that invites the class to reason about your findings. Anticipate questions about experimental controls, assumptions and alternative explanations.

## Preliminaries: environment

The course environment provides all libraries that you need to execute the startet codes (see [ReadMe for intallation instructions](https://github.com/stefanlessmann/ESMT_IML)). You can install additional libraries if needed. 

Run the supplied cells first. They provide a starting point, **not a completed tutorial**. Code cells marked `YOUR GROUP'S WORK` are intentionally unfinished and can be replaced or expanded.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 888
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.size': 11})
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')

## Preliminaries: supplied data and helpers

There are 1,600 development cases and 400 future cases. Chronological validation uses the first 1,200 development cases for training and the next 400 for validation. Random splitting uses the same counts. The market shift begins at case 1,100. Cases are ordered in time, but their random disturbances are independent. The model is intentionally misspecified because it omits the market shift; discuss this assumption explicitly.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

def make_temporal_cases(seed=SEED, shift_size=8.0, shift_at=1100):
    rng = np.random.default_rng(seed)
    n = 2000
    t = np.arange(n)
    x = rng.normal(0, 1, n)
    market_effect = shift_size * (t >= shift_at)
    target = 10 + 1.8 * x + market_effect + rng.normal(0, 0.8, n)
    return pd.DataFrame({'time': t, 'x': x, 'target': target,
                         'market_effect_for_diagnostics': market_effect})

def split_development(frame):
    return frame.iloc[:1600].copy(), frame.iloc[1600:].copy()

def validation_indices(n=1600, chronological=False, seed=SEED):
    if chronological:
        return np.arange(n - 400), np.arange(n - 400, n)
    return train_test_split(np.arange(n), test_size=400, random_state=seed)

def fresh_model():
    return LinearRegression()

cases = make_temporal_cases()
development, future = split_development(cases)
FEATURES = ['x']  # Time and generator diagnostics are not model features.
TARGET = 'target'
display(development.head())
print('Development cases:', len(development), '| Held-back future cases:', len(future))

## Investigation 1: Make the timing assumptions visible

Plot development targets and identify the change point. Draw or plot which cases each split places in training and validation. State the intended deployment population before reporting errors.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 2: Run the validation comparison

Fit a fresh model per split. Compare MAE/RMSE for chronological validation and several random splits. Verify equal sample counts and explain why the fitted training populations differ.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 3: Check the deployment conclusion

Refit once on the full development data and evaluate on future. Repeat the planned protocol with shift_size=0 as a contrasting condition. Explain the limits of comparing validation estimates with a refitted final model.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Your teaching conclusions

Replace this text with:

1. A direct answer to the tutorial's central question, supported by your results.
2. Two or three takeaways that classmates should remember.
3. One limitation, counterexample or condition under which the result changes.
4. One discussion question, plus notes on how your group would answer it.

## Contribution and verification record

- **Supplied material retained or changed:** …
- **Your group's experimental and explanatory contributions:** …
- **Independent check of an important result:** …
- **AI tools and their contributions, if any:** …
- **How generated code/explanations were checked:** …
- **Package versions, random seeds and reproduction instructions:** …
- **Group members and contributions:** …

## Before presenting

- [ ] Restarted the kernel and ran all cells successfully.
- [ ] Checked that splits, metrics, units and class labels are correct.
- [ ] Distinguished evidence from interpretation; avoided claims stronger than the experiment supports.
- [ ] Labelled plots and made figures legible for classmates.
- [ ] Rehearsed within 15 minutes and prepared for discussion.
- [ ] Included the completed notebook and any additional presentation material.